In [2]:
!pip install -qU langchain langchain-community langchain-core langchain-google-genai faiss-cpu pypdf
!pip install -U chromadb langchain langchain-community langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [3]:

from google.colab import userdata
key=userdata.get('Key_G1')
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview", google_api_key=key)

from langchain_community.document_loaders import PyPDFLoader
pdf=PyPDFLoader("/content/sample_data/9 pages.history.pdf")
pdf_loader=pdf.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100,separators=["\n\n","\n"," "])
splitting=splitter.split_documents(pdf_loader)

from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-001",google_api_key=key)

from langchain_chroma import Chroma
vector_st=Chroma.from_documents(documents=splitting,embedding=embeddings)
vector_ind=vector_st.as_retriever(search_kwargs={"k":4})


In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

template = """Answer the question based only on the following PDF:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)
rag_chain = (
    {"context": vector_ind | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

response = rag_chain.invoke("Summarize the key historical events mentioned in the document.?")
print(response)

Based on the provided document, the key historical events are categorized into two phases of the Indian armed forces’ journey:

**Early Events and Phase I (1919–1962)**
*   **1919:** The Jallianwala Bagh massacre, where British forces used Gorkha and Baluchi troops against civilians.
*   **1947–1948:** The war in Kashmir, marking the start of the first phase of the armed forces' journey.
*   **Korean War:** India participated by providing medical assistance and sending the 60th Parachute Field Ambulance Platoon.
*   **1950:** China’s entry into Tibet.
*   **1954:** The signing of a treaty between India and the Tibet Region of China.
*   **1959:** Bloody clashes occurred at Longju (August) and an ambush of an Indian police party at Konka La in eastern Ladakh (October).
*   **1962:** The India–China War, described as a "debacle" for India, which concluded the first phase and led to significant changes in military equipping and disposition.

**Phase II (1962–1988)**
*   **Post-1962:** The

In [5]:
!pip install grandalf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.9 MB/s eta 0:00:00


# **Workflow**

In [6]:
print(rag_chain.get_graph().print_ascii())

              +---------------------------------+           
              | Parallel<context,question>Input |           
              +---------------------------------+           
                    ****                ****                
                 ***                        ***             
               **                              ***          
+----------------------+                          **        
| VectorStoreRetriever |                           *        
+----------------------+                           *        
            *                                      *        
            *                                      *        
            *                                      *        
    +-------------+                         +-------------+ 
    | format_docs |                         | Passthrough | 
    +-------------+*                        +-------------+ 
                    ****                ****                
                        